In [ ]:
#三种模型整体预测性能评估
#方案一：三种模型一起运行并呈现结果，可能比较费电，如果电脑不支持可以考虑采用下面的方案二，分别跑三种模型，并手动输入跑出的数据呈现结果

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet, Lasso
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker

# ==========================================
# 全局字体设置：将所有图表字体改为 Times New Roman
# ==========================================
plt.rcParams['font.family'] = 'Times New Roman'
# plt.rcParams['axes.unicode_minus'] = False # 如果遇到负号显示为方块，可取消注释此行

# 数据加载与预处理
def load_and_prepare_data(file_path='../data/MERGED_DATA_Winsor.dta'): #在此更换数据路径
    print("正在加载数据...")
    df = pd.read_stata(file_path)   
    print(f"数据形状: {df.shape}")
    print(f"年份范围: {df['year'].min()} - {df['year'].max()}")
    print(f"企业数量: {df['stock'].nunique()}")
    
    return df

# 特征列：每行为一类特征
feature_cols = [
    'TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG',
    'CEORisk', 'CEODegree', 'IsMBA', 'FinBack', 'CEOAcademicBack', 'Versatility', 'Top3ManageSumSalary', 'CEOIncentive', 'REM', 'IntControl', 'KV', 'Deficiency',
    'EmployeeTurnover', 'ESOP', 'Paygap', 'SocInsurance', 'EmployeeSalary',
    'InvestorShare', 'Ownstructure', 'MajorShareholders', 'Ownership', 'Litigation', 'VAM',
    'Boardsize', 'IndDirectorRatio', 'FemaleDirector', 'DO', 'DCG', 'AISkill', 'DataSkill', 'ChairAcademicBack',
    'FC', 'BOND_D', 'BOND_P', 'BankHHI', 'Top3Bank', 'BankCR',
    'AnaAttention', 'FinNews', 'NegFinNews', 'FERR', 'FOPT', 
    'Financeavailable', 'Shortavailable', 'Liquidity', 'NCSKEW', 'DUVOL', 'SC',
    'GovSub', 'Dige', 'Gdatact', 'Ipp', 'HTZone', 'EnvSubsidy', 'Enrelcost', 'TaxRate', 'Talent', 'IP_EncImp',
    'HHI', 'Zombie_Ratio',
    'SupplierHHI', 'CustomerHHI'
]

def prepare_year_data(data, train_years, test_year, target_col='RDCost_t1'): #在此更换目标变量
    """
    准备训练集和测试集数据
    """
    # 样本筛选
    complete_data = data[data['complete_obs'] == 0].copy()
    
    # 从筛选后的数据中划分训练集和测试集
    train_data = complete_data[complete_data['year'].isin(train_years)].copy() 
    test_data = complete_data[complete_data['year'] == test_year].copy()
    
    X_train = train_data[feature_cols].copy()
    y_train = train_data[target_col].copy()
    X_test = test_data[feature_cols].copy()
    y_test = test_data[target_col].copy()
    
    # 处理缺失值和异常值 - 统一处理方式
    for col in feature_cols:
        X_train[col] = X_train[col].replace([np.inf, -np.inf], np.nan)
        X_test[col] = X_test[col].replace([np.inf, -np.inf], np.nan)
    
    train_mask = y_train.notna() & X_train.notna().all(axis=1)
    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    
    test_mask = y_test.notna() & X_test.notna().all(axis=1)
    X_test_clean = X_test[test_mask]
    y_test_clean = y_test[test_mask]
    
    print(f"训练集: {len(X_train_clean)} 个样本 (年份: {train_years})")
    print(f"测试集: {len(X_test_clean)} 个样本 (年份: {test_year})")
    
    # 标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled = scaler.transform(X_test_clean)
    
    return X_train_scaled, y_train_clean.values, X_test_scaled, y_test_clean.values, X_test_clean, y_test_clean

def time_series_evaluation(X_train, y_train, X_test, y_test, model_name='GBDT'):
    """
    时间序列划分的模型评估 - 加入网格搜索
    """
    if len(X_train) < 10 or len(X_test) < 5:
        print("样本数量不足，跳过该划分")
        return None
    
    try:
        # 设置默认的交叉验证折数
        cv_folds = 5 
        
        if model_name == 'GBDT':
            # GBDT模型参数
            model = GradientBoostingRegressor(random_state=42)
            param_grid = {
                'learning_rate': [0.1, 0.2],
                'n_estimators': [100, 150],
                'max_depth': [3, 5]
            }
            
        elif model_name == 'RF':
            # 随机森林模型参数（已根据需求更新）
            model = RandomForestRegressor(random_state=42, n_jobs=-1)
            param_grid = {
                'n_estimators': [100, 150],
                'max_depth': [5, 8, 10],            
                'max_features': ['sqrt', 1.0]       
            }
            cv_folds = 3  # 为随机森林单独设定交叉验证折数为3
            
        elif model_name == 'OLS':
            # OLS模型 - 直接训练，不需要网格搜索
            ols_model = LinearRegression()
            ols_model.fit(X_train, y_train)
            y_pred = ols_model.predict(X_test)
            
            r2 = r2_score(y_test, y_pred)
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mae = np.mean(np.abs(y_test - y_pred))
            
            results = {
                'r2': r2,
                'mse': mse,
                'rmse': rmse,
                'mae': mae,
                'n_train': len(X_train),
                'n_test': len(X_test),
                'model': ols_model,
                'best_params': 'OLS (无超参数)'
            }
            
            return results

        # GBDT和随机森林模型使用网格搜索
        print(f"正在进行{model_name}网格搜索...")
        grid_search = GridSearchCV(model, param_grid, 
                                cv=cv_folds, scoring='r2', 
                                n_jobs=-1, verbose=0)
        
        grid_search.fit(X_train, y_train)
        print(f"最优参数: {grid_search.best_params_}")
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)
        
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(y_test - y_pred))
        
        results = {
            'r2': r2,
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'n_train': len(X_train),
            'n_test': len(X_test),
            'model': best_model,
            'best_params': grid_search.best_params_
        }
        
        return results
        
    except Exception as e:
        print(f"模型训练失败: {e}")
        return None

def plot_r2_boxplot(all_results):
    """
    绘制三种模型R²的箱线图
    改进点：Y轴范围仅基于'可见'数据（排除被隐藏的异常值），消除底部留白
    """
    r2_data = []
    
    # 1. 收集数据
    for model_name in ['GBDT', 'RF', 'OLS']:
        if model_name in all_results and all_results[model_name]['detailed_results']:
            detailed_results = all_results[model_name]['detailed_results']
            r2_values = [result['r2'] for result in detailed_results]
            for r2 in r2_values:
                r2_data.append({'Model': model_name, 'R²': r2})
    
    if not r2_data:
        print("没有足够的数据绘制箱线图")
        return
    
    r2_df = pd.DataFrame(r2_data)
    
    plt.figure(figsize=(10, 6))
    colors = ['red', 'orange', 'yellow']
    
    # 2. 绘制箱线图
    box_plot = sns.boxplot(data=r2_df, x='Model', y='R²', palette=colors, showfliers=False)
    
    # 3. 绘制均值点
    means = []
    for model in ['GBDT', 'RF', 'OLS']:
        model_data = r2_df[r2_df['Model'] == model]['R²']
        if len(model_data) > 0:
            mean_val = model_data.mean()
            means.append(mean_val)
        else:
            means.append(None)

    for i, mean_val in enumerate(means):
        if mean_val is not None:
            plt.plot(i, mean_val, 'o', 
                     markersize=6, 
                     markeredgewidth=1, 
                     markeredgecolor='black',
                     markerfacecolor='white') 
    
    Q1 = r2_df['R²'].quantile(0.25)
    Q3 = r2_df['R²'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    visible_data = r2_df[(r2_df['R²'] >= lower_bound) & (r2_df['R²'] <= upper_bound)]
    
    # 获取可见数据的最大最小值
    y_min_visible = visible_data['R²'].min()
    y_max_visible = visible_data['R²'].max()
    
    # 给予一点点缓冲空间 (比如 0.05)
    padding = 0.05
    
    # 设置 Y 轴范围
    plt.ylim(y_min_visible - padding, y_max_visible + padding)
    
    # 强制刻度为 0.1
    ax = plt.gca()
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
    
    # 设置标签
    plt.xlabel('Model', fontsize=12)
    plt.ylabel('R²', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.title('Time-Series Dispersion of R² - R&D Cost', fontsize=14, pad=15)
    
    plt.tight_layout()
    plt.show()
    
def print_performance_table(all_results):
    """输出三种模型性能对比表"""
    metrics = ['MSE', 'RMSE', 'MAE', 'R²']
    models = ['GBDT', 'RF', 'OLS']
    
    # 收集数据
    performance_data = []
    
    for model_name in models:
        if model_name in all_results and all_results[model_name]['detailed_results']:
            detailed_results = all_results[model_name]['detailed_results']
            
            avg_mse = np.mean([result['mse'] for result in detailed_results])
            avg_rmse = np.mean([result['rmse'] for result in detailed_results])
            avg_mae = np.mean([result['mae'] for result in detailed_results])
            avg_r2 = np.mean([result['r2'] for result in detailed_results])
            
            performance_data.append({
                'Model': model_name,
                'MSE': avg_mse,
                'RMSE': avg_rmse,
                'MAE': avg_mae,
                'R²': avg_r2
            })
    
    if not performance_data:
        print("没有性能数据可显示")
        return
    
    # 创建对比表
    performance_df = pd.DataFrame(performance_data)
    performance_df.set_index('Model', inplace=True)
    
    print("\n" + "="*80)
    print("三种模型性能对比表")
    print("="*80)
    print(f"{'指标':<10} {'GBDT':<12} {'RF':<12} {'OLS':<12}")
    print("-" * 50)
    
    for metric in metrics:
        gbdt_val = performance_df.loc['GBDT', metric] if 'GBDT' in performance_df.index else 'N/A'
        rf_val = performance_df.loc['RF', metric] if 'RF' in performance_df.index else 'N/A'
        ols_val = performance_df.loc['OLS', metric] if 'OLS' in performance_df.index else 'N/A'
        
        if metric != 'R²':
            print(f"{metric:<10} {gbdt_val:<12.4f} {rf_val:<12.4f} {ols_val:<12.4f}")
        else:
            print(f"{metric:<10} {gbdt_val:<12.4f} {rf_val:<12.4f} {ols_val:<12.4f}")

def main():
    """
    主函数
    """
    # 加载数据
    data = load_and_prepare_data()
    
    # 进行时间序列分析 - 三种模型对比
    print("\n开始时间序列分析...")
    
    # 使用三种模型分别进行分析
    models = ['GBDT', 'RF', 'OLS']
    all_results = {}
    
    for model_name in models:
        print(f"\n{'='*80}")
        print(f"开始分析 {model_name} 模型")
        print('='*80)
        
        yearly_results = {}
        detailed_results = []
        
        # 定义训练集和测试集的划分方案
        train_test_splits = [
            ([2012, 2013, 2014], 2016),
            ([2012, 2013, 2014, 2015], 2017),
            ([2012, 2013, 2014, 2015, 2016], 2018),
            ([2012, 2013, 2014, 2015, 2016, 2017], 2019),
            ([2012, 2013, 2014, 2015, 2016, 2017, 2018], 2020),
            ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], 2021),
            ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], 2022),
            ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021], 2023)
        ]
        
        for i, (train_years, test_year) in enumerate(train_test_splits, 1):
            print(f"\n划分 {i}: 训练集年份 {train_years} → 测试集年份 {test_year}")
            
            # 准备数据
            X_train, y_train, X_test, y_test, X_test_orig, y_test_orig = prepare_year_data(
                data, train_years, test_year
            )
            
            # 评估模型
            results = time_series_evaluation(X_train, y_train, X_test, y_test, model_name)
            
            if results is not None:
                yearly_results[f"{train_years[0]}-{train_years[-1]}_to_{test_year}"] = results
                detailed_results.append(results)
                
                print(f"R²: {results['r2']:.4f}, MSE: {results['mse']:.4f}, RMSE: {results['rmse']:.4f}, MAE: {results['mae']:.4f}")
            else:
                print("该划分无法评估")
        
        all_results[model_name] = {
            'yearly_results': yearly_results,
            'detailed_results': detailed_results
        }
    
    # ==================== 输出主要结果 ====================
    
    # 1. 每次训练测试划分的结果（已经在上面输出）
    
    # 2. 三种模型样本外R方的箱线图
    print("\n" + "="*80)
    print("2. 三种模型样本外R²箱线图")
    print("="*80)
    plot_r2_boxplot(all_results)
    
    # 3. 三种模型参数对比表
    print("\n" + "="*80)
    print("3. 三种模型性能对比表")
    print("="*80)
    print_performance_table(all_results)
    
    return all_results

if __name__ == "__main__":
    all_results = main()

In [ ]:
#整体预测性能评估方案二：分模型评估
#GBDT整体预测性能评估

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# 数据加载与预处理
def load_and_prepare_data(file_path='../data/MERGED_DATA_Winsor.dta'):#在此更换数据路径
    print("正在加载数据...")
    df = pd.read_stata(file_path)   
    print(f"数据形状: {df.shape}")
    print(f"年份范围: {df['year'].min()} - {df['year'].max()}")
    print(f"企业数量: {df['stock'].nunique()}")
    
    return df

# 特征列：每行为一类特征
feature_cols = [
    'TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG',
    'CEORisk', 'CEODegree', 'IsMBA', 'FinBack', 'CEOAcademicBack', 'Versatility', 'Top3ManageSumSalary', 'CEOIncentive', 'REM', 'IntControl', 'KV', 'Deficiency',
    'EmployeeTurnover', 'ESOP', 'Paygap', 'SocInsurance', 'EmployeeSalary',
    'InvestorShare', 'Ownstructure', 'MajorShareholders', 'Ownership', 'Litigation', 'VAM',
    'Boardsize', 'IndDirectorRatio', 'FemaleDirector', 'DO', 'DCG', 'AISkill', 'DataSkill', 'ChairAcademicBack',
    'FC', 'BOND_D', 'BOND_P', 'BankHHI', 'Top3Bank', 'BankCR',
    'AnaAttention', 'FinNews', 'NegFinNews', 'FERR', 'FOPT', 
    'Financeavailable', 'Shortavailable', 'Liquidity', 'NCSKEW', 'DUVOL', 'SC',
    'GovSub', 'Dige', 'Gdatact', 'Ipp', 'HTZone', 'EnvSubsidy', 'Enrelcost', 'TaxRate', 'Talent', 'IP_EncImp',
    'HHI', 'Zombie_Ratio',
    'SupplierHHI', 'CustomerHHI'
]

def prepare_year_data(data, train_years, test_year, target_col='RDCost_t1'):#在此更换目标变量
    """
    准备训练集和测试集数据
    """
    # 样本筛选
    complete_data = data[data['complete_obs'] == 0].copy()
    
    # 从筛选后的数据中划分训练集和测试集
    train_data = complete_data[complete_data['year'].isin(train_years)].copy() 
    test_data = complete_data[complete_data['year'] == test_year].copy()
    
    X_train = train_data[feature_cols].copy()
    y_train = train_data[target_col].copy()
    X_test = test_data[feature_cols].copy()
    y_test = test_data[target_col].copy()
    
    # 处理缺失值和异常值 - 统一处理方式
    for col in feature_cols:
        X_train[col] = X_train[col].replace([np.inf, -np.inf], np.nan)
        X_test[col] = X_test[col].replace([np.inf, -np.inf], np.nan)
    
    train_mask = y_train.notna() & X_train.notna().all(axis=1)
    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    test_mask = y_test.notna() & X_test.notna().all(axis=1)
    X_test_clean = X_test[test_mask]
    y_test_clean = y_test[test_mask]
    
    print(f"训练集: {len(X_train_clean)} 个样本 (年份: {train_years})")
    print(f"测试集: {len(X_test_clean)} 个样本 (年份: {test_year})")
    
    # 标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled = scaler.transform(X_test_clean)
    
    return X_train_scaled, y_train_clean.values, X_test_scaled, y_test_clean.values, X_test_clean, y_test_clean

def time_series_evaluation(X_train, y_train, X_test, y_test, model_name='GBDT'):
    """
    时间序列划分的模型评估
    """
    if len(X_train) < 10 or len(X_test) < 5:
        print("样本数量不足，跳过该划分")
        return None
    
    try:
        # 使用网格搜索寻找最优参数
        gbdt = GradientBoostingRegressor(random_state=42)
        gbdt_param_grid = {
            'learning_rate': [0.1, 0.2],
            'n_estimators': [100, 150],
            'max_depth': [3, 5]
        }
        
        print("正在进行网格搜索...")
        gbdt_grid = GridSearchCV(gbdt, gbdt_param_grid, 
                                cv=5, scoring='r2',
                                n_jobs=-1, verbose=0) 
        
        gbdt_grid.fit(X_train, y_train)
        print(f"最优参数: {gbdt_grid.best_params_}")
        best_model = gbdt_grid.best_estimator_
        y_pred = best_model.predict(X_test)
        
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(y_test - y_pred))
        
        results = {
            'r2': r2,
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'n_train': len(X_train),
            'n_test': len(X_test),
            'model': best_model,
            'best_params': gbdt_grid.best_params_  # 保存最优参数
        }
        
        return results
        
    except Exception as e:
        print(f"模型训练失败: {e}")
        return None

def yearly_time_series_analysis(data, model_name='GBDT'):
    """
    按年份划分训练集和测试集的时间序列分析
    """
    yearly_results = {}
    
    # 定义训练集和测试集的划分方案
    train_test_splits = [
        ([2012, 2013, 2014], 2016),
        ([2012, 2013, 2014, 2015], 2017),
        ([2012, 2013, 2014, 2015, 2016], 2018),
        ([2012, 2013, 2014, 2015, 2016, 2017], 2019),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018], 2020),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], 2021),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], 2022),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021], 2023)
    ]
    
    print(f"\n开始时间序列分析 - 使用GBDT模型（带网格搜索）")
    print("=" * 80)
    
    # 存储所有划分的详细结果
    detailed_results = []
    
    for i, (train_years, test_year) in enumerate(train_test_splits, 1):
        
        # 准备数据
        X_train, y_train, X_test, y_test, X_test_orig, y_test_orig = prepare_year_data(
            data, train_years, test_year
        )
        
        # 评估模型
        results = time_series_evaluation(X_train, y_train, X_test, y_test, model_name)
        
        if results is not None:
            yearly_results[f"{train_years[0]}-{train_years[-1]}_to_{test_year}"] = results
            
            # 存储详细结果用于后续汇总
            detailed_results.append({
                'split': f"{train_years[0]}-{train_years[-1]}_to_{test_year}",
                'r2': results['r2'],
                'mse': results['mse'],
                'rmse': results['rmse'],
                'mae': results['mae'],
                'n_train': results['n_train'],
                'n_test': results['n_test'],
                'best_params': results['best_params']  # 保存最优参数
            })
            
            print(f"R²: {results['r2']:.4f}, MSE: {results['mse']:.4f}, RMSE: {results['rmse']:.4f}, MAE: {results['mae']:.4f}")
        else:
            print("该划分无法评估")
    
    # 输出每种划分的详细结果
    print(f"\n{'='*80}")
    print("每种训练-测试划分的详细结果:")
    print(f"{'划分':<25} {'R²':<8} {'MSE':<10} {'RMSE':<8} {'MAE':<8} {'训练样本':<10} {'测试样本':<10}")
    print("-" * 80)
    
    for result in detailed_results:
        print(f"{result['split']:<25} {result['r2']:.4f}  {result['mse']:.4f}    {result['rmse']:.4f}   {result['mae']:.4f}   {result['n_train']:<10} {result['n_test']:<10}")
    
    # 计算整体平均性能
    if detailed_results:
        mean_r2 = np.mean([result['r2'] for result in detailed_results])
        mean_mse = np.mean([result['mse'] for result in detailed_results])
        mean_rmse = np.mean([result['rmse'] for result in detailed_results])
        mean_mae = np.mean([result['mae'] for result in detailed_results])
        
        print(f"\n{'='*80}")
        print("整体平均性能汇总:")
        print(f"{'R²':<8} {mean_r2:.4f}")
        print(f"{'MSE':<8} {mean_mse:.4f}")
        print(f"{'RMSE':<8} {mean_rmse:.4f}")
        print(f"{'MAE':<8} {mean_mae:.4f}")
        
        print(f"\n有效划分数量: {len(detailed_results)}")
    
    return yearly_results, detailed_results

def main():
    """
    主函数
    """
    # 加载数据
    data = load_and_prepare_data()
    
    # 进行时间序列分析
    print("\n开始时间序列分析...")
    
    # 使用GBDT模型（带网格搜索）
    gbdt_results, detailed_results = yearly_time_series_analysis(data, model_name='GBDT')
    
    return gbdt_results, detailed_results

if __name__ == "__main__":
    results, detailed_results = main()


In [ ]:
#整体预测性能评估方案二：分模型评估
#随机森林整体预测性能评估

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# 数据加载与预处理
def load_and_prepare_data(file_path='../data/MERGED_DATA_Winsor.dta'):#在此更换数据路径
    print("正在加载数据...")
    df = pd.read_stata(file_path)   
    print(f"数据形状: {df.shape}")
    print(f"年份范围: {df['year'].min()} - {df['year'].max()}")
    print(f"企业数量: {df['stock'].nunique()}")
    
    return df

# 特征列：每行为一类特征
feature_cols = [
    'TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG',
    'CEORisk', 'CEODegree', 'IsMBA', 'FinBack', 'CEOAcademicBack', 'Versatility', 'Top3ManageSumSalary', 'CEOIncentive', 'REM', 'IntControl', 'KV', 'Deficiency',
    'EmployeeTurnover', 'ESOP', 'Paygap', 'SocInsurance', 'EmployeeSalary',
    'InvestorShare', 'Ownstructure', 'MajorShareholders', 'Ownership', 'Litigation', 'VAM',
    'Boardsize', 'IndDirectorRatio', 'FemaleDirector', 'DO', 'DCG', 'AISkill', 'DataSkill', 'ChairAcademicBack',
    'FC', 'BOND_D', 'BOND_P', 'BankHHI', 'Top3Bank', 'BankCR',
    'AnaAttention', 'FinNews', 'NegFinNews', 'FERR', 'FOPT', 
    'Financeavailable', 'Shortavailable', 'Liquidity', 'NCSKEW', 'DUVOL', 'SC',
    'GovSub', 'Dige', 'Gdatact', 'Ipp', 'HTZone', 'EnvSubsidy', 'Enrelcost', 'TaxRate', 'Talent', 'IP_EncImp',
    'HHI', 'Zombie_Ratio',
    'SupplierHHI', 'CustomerHHI'
]

def prepare_year_data(data, train_years, test_year, target_col='RDCost_t1'):#在此更换目标变量
    """
    准备训练集和测试集数据
    """
    # 样本筛选
    complete_data = data[data['complete_obs'] == 0].copy()
    
    # 从筛选后的数据中划分训练集和测试集
    train_data = complete_data[complete_data['year'].isin(train_years)].copy() 
    test_data = complete_data[complete_data['year'] == test_year].copy()
    
    X_train = train_data[feature_cols].copy()
    y_train = train_data[target_col].copy()
    X_test = test_data[feature_cols].copy()
    y_test = test_data[target_col].copy()
    
    # 处理缺失值和异常值 - 统一处理方式
    for col in feature_cols:
        X_train[col] = X_train[col].replace([np.inf, -np.inf], np.nan)
        X_test[col] = X_test[col].replace([np.inf, -np.inf], np.nan)
    
    train_mask = y_train.notna() & X_train.notna().all(axis=1)
    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    test_mask = y_test.notna() & X_test.notna().all(axis=1)
    X_test_clean = X_test[test_mask]
    y_test_clean = y_test[test_mask]
    
    print(f"训练集: {len(X_train_clean)} 个样本 (年份: {train_years})")
    print(f"测试集: {len(X_test_clean)} 个样本 (年份: {test_year})")
    
    # 标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled = scaler.transform(X_test_clean)
    
    return X_train_scaled, y_train_clean.values, X_test_scaled, y_test_clean.values, X_test_clean, y_test_clean

def time_series_evaluation(X_train, y_train, X_test, y_test, model_name='RandomForest'):
    """
    时间序列划分的模型评估
    """
    if len(X_train) < 10 or len(X_test) < 5:
        print("样本数量不足，跳过该划分")
        return None
    
    try:
        # 使用网格搜索寻找最优参数
        rf = RandomForestRegressor(random_state=42, n_jobs=-1)
        
        rf_param_grid = {
            'n_estimators': [100, 150],
            'max_depth': [5, 8, 10],            
            'max_features': ['sqrt', 1.0]      
        }
        
        print("正在进行网格搜索...")
        rf_grid = GridSearchCV(rf, rf_param_grid, 
                              cv=3, scoring='r2', 
                              n_jobs=-1, verbose=0)
        
        rf_grid.fit(X_train, y_train)
        print(f"最优参数: {rf_grid.best_params_}")
        best_model = rf_grid.best_estimator_
        y_pred = best_model.predict(X_test)
                
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(y_test - y_pred))
        
        results = {
            'r2': r2,
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'n_train': len(X_train),
            'n_test': len(X_test),
            'model': best_model,
            'best_params': rf_grid.best_params_  # 保存最优参数
        }
        
        return results
        
    except Exception as e:
        print(f"模型训练失败: {e}")
        return None

def yearly_time_series_analysis(data, model_name='RandomForest'):
    """
    按年份划分训练集和测试集的时间序列分析
    """
    yearly_results = {}
    
    # 定义训练集和测试集的划分方案
    train_test_splits = [
        ([2012, 2013, 2014], 2016),
        ([2012, 2013, 2014, 2015], 2017),
        ([2012, 2013, 2014, 2015, 2016], 2018),
        ([2012, 2013, 2014, 2015, 2016, 2017], 2019),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018], 2020),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], 2021),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], 2022),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021], 2023)
    ]
    
    print(f"\n开始时间序列分析 - 使用随机森林模型（带网格搜索）")
    print("=" * 80)
    
    # 存储所有划分的详细结果
    detailed_results = []
    
    for i, (train_years, test_year) in enumerate(train_test_splits, 1):
        
        # 准备数据
        X_train, y_train, X_test, y_test, X_test_orig, y_test_orig = prepare_year_data(
            data, train_years, test_year
        )
        
        # 评估模型
        results = time_series_evaluation(X_train, y_train, X_test, y_test, model_name)
        
        if results is not None:
            yearly_results[f"{train_years[0]}-{train_years[-1]}_to_{test_year}"] = results
            
            # 存储详细结果用于后续汇总
            detailed_results.append({
                'split': f"{train_years[0]}-{train_years[-1]}_to_{test_year}",
                'r2': results['r2'],
                'mse': results['mse'],
                'rmse': results['rmse'],
                'mae': results['mae'],
                'n_train': results['n_train'],
                'n_test': results['n_test'],
                'best_params': results['best_params']  # 保存最优参数
            })
            
            print(f"R²: {results['r2']:.4f}, MSE: {results['mse']:.4f}, RMSE: {results['rmse']:.4f}, MAE: {results['mae']:.4f}")
        else:
            print("该划分无法评估")
    
    # 输出每种划分的详细结果
    print(f"\n{'='*80}")
    print("每种训练-测试划分的详细结果:")
    print(f"{'划分':<25} {'R²':<8} {'MSE':<10} {'RMSE':<8} {'MAE':<8} {'训练样本':<10} {'测试样本':<10}")
    print("-" * 80)
    
    for result in detailed_results:
        print(f"{result['split']:<25} {result['r2']:.4f}  {result['mse']:.4f}    {result['rmse']:.4f}   {result['mae']:.4f}   {result['n_train']:<10} {result['n_test']:<10}")
    
    # 计算整体平均性能
    if detailed_results:
        mean_r2 = np.mean([result['r2'] for result in detailed_results])
        mean_mse = np.mean([result['mse'] for result in detailed_results])
        mean_rmse = np.mean([result['rmse'] for result in detailed_results])
        mean_mae = np.mean([result['mae'] for result in detailed_results])
        
        print(f"\n{'='*80}")
        print("整体平均性能汇总:")
        print(f"{'R²':<8} {mean_r2:.4f}")
        print(f"{'MSE':<8} {mean_mse:.4f}")
        print(f"{'RMSE':<8} {mean_rmse:.4f}")
        print(f"{'MAE':<8} {mean_mae:.4f}")
        
        print(f"\n有效划分数量: {len(detailed_results)}")
    
    return yearly_results, detailed_results

def main():
    """
    主函数
    """
    # 加载数据
    data = load_and_prepare_data()
    
    # 进行时间序列分析
    print("\n开始时间序列分析...")
    
    # 使用随机森林模型（带网格搜索）
    rf_results, detailed_results = yearly_time_series_analysis(data, model_name='RandomForest')
    
    return rf_results, detailed_results

if __name__ == "__main__":
    results, detailed_results = main()

In [ ]:
#整体预测性能评估方案二：分模型评估
#OLS整体预测性能评估

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import KFold
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# 数据加载与预处理
def load_and_prepare_data(file_path='../data/MERGED_DATA_Winsor.dta'):#在此更换数据路径
    print("正在加载数据...")
    df = pd.read_stata(file_path)   
    print(f"数据形状: {df.shape}")
    print(f"年份范围: {df['year'].min()} - {df['year'].max()}")
    print(f"企业数量: {df['stock'].nunique()}")
    
    return df

# 特征列：每行为一类特征，可更换进行预测
feature_cols = [
    'TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG',
    'CEORisk', 'CEODegree', 'IsMBA', 'FinBack', 'CEOAcademicBack', 'Versatility', 'Top3ManageSumSalary', 'CEOIncentive', 'REM', 'IntControl', 'KV', 'Deficiency',
    'EmployeeTurnover', 'ESOP', 'Paygap', 'SocInsurance', 'EmployeeSalary',
    'InvestorShare', 'Ownstructure', 'MajorShareholders', 'Ownership', 'Litigation', 'VAM',
    'Boardsize', 'IndDirectorRatio', 'FemaleDirector', 'DO', 'DCG', 'AISkill', 'DataSkill', 'ChairAcademicBack',
    'FC', 'BOND_D', 'BOND_P', 'BankHHI', 'Top3Bank', 'BankCR',
    'AnaAttention', 'FinNews', 'NegFinNews', 'FERR', 'FOPT', 
    'Financeavailable', 'Shortavailable', 'Liquidity', 'NCSKEW', 'DUVOL', 'SC',
    'GovSub', 'Dige', 'Gdatact', 'Ipp', 'HTZone', 'EnvSubsidy', 'Enrelcost', 'TaxRate', 'Talent', 'IP_EncImp',
    'HHI', 'Zombie_Ratio',
    'SupplierHHI', 'CustomerHHI'
]

def prepare_year_data(data, train_years, test_year, target_col='RDCost_t1'):#在此更换目标变量
    """
    准备训练集和测试集数据
    """
    # 样本筛选
    # 如果target_col为'RDCost_t1_w'，筛选complete_obs==0；如果target_col为'InvPatent_t1'和'Patent_t1'，筛选complete_obs1==0
    complete_data = data[data['complete_obs1'] == 0].copy()
    
    # 从筛选后的数据中划分训练集和测试集
    train_data = complete_data[complete_data['year'].isin(train_years)].copy() 
    test_data = complete_data[complete_data['year'] == test_year].copy()
    
    X_train = train_data[feature_cols].copy()
    y_train = train_data[target_col].copy()
    X_test = test_data[feature_cols].copy()
    y_test = test_data[target_col].copy()
    
    # 处理缺失值和异常值 - 统一处理方式
    for col in feature_cols:
        X_train[col] = X_train[col].replace([np.inf, -np.inf], np.nan)
        X_test[col] = X_test[col].replace([np.inf, -np.inf], np.nan)
    
    train_mask = y_train.notna() & X_train.notna().all(axis=1)
    X_train_clean = X_train[train_mask]
    y_train_clean = y_train[train_mask]
    test_mask = y_test.notna() & X_test.notna().all(axis=1)
    X_test_clean = X_test[test_mask]
    y_test_clean = y_test[test_mask]
    
    print(f"训练集: {len(X_train_clean)} 个样本 (年份: {train_years})")
    print(f"测试集: {len(X_test_clean)} 个样本 (年份: {test_year})")
    
    # 标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_clean)
    X_test_scaled = scaler.transform(X_test_clean)
    
    return X_train_scaled, y_train_clean.values, X_test_scaled, y_test_clean.values, X_test_clean, y_test_clean

def time_series_evaluation(X_train, y_train, X_test, y_test, model_name='OLS'):
    """
    时间序列划分的模型评估
    """
    if len(X_train) < 10 or len(X_test) < 5:
        print("样本数量不足，跳过该划分")
        return None
    
    try:
        ols_model = LinearRegression()
        ols_model.fit(X_train, y_train)
        y_pred = ols_model.predict(X_test)
        
        r2 = r2_score(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        mae = np.mean(np.abs(y_test - y_pred))
        
        results = {
            'r2': r2,
            'mse': mse,
            'rmse': rmse,
            'mae': mae,
            'n_train': len(X_train),
            'n_test': len(X_test),
        }
        
        return results
        
    except Exception as e:
        print(f"模型训练失败: {e}")
        return None

def yearly_time_series_analysis(data, model_name='OLS'):
    """
    按年份划分训练集和测试集的时间序列分析
    """
    yearly_results = {}
    
    # 定义训练集和测试集的划分方案
    train_test_splits = [
        ([2012, 2013, 2014], 2016),
        ([2012, 2013, 2014, 2015], 2017),
        ([2012, 2013, 2014, 2015, 2016], 2018),
        ([2012, 2013, 2014, 2015, 2016, 2017], 2019),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018], 2020),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], 2021),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], 2022),
        ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021], 2023)
    ]
    
    print(f"\n开始时间序列分析 - 使用OLS模型（带网格搜索）")
    print("=" * 80)
    
    # 存储所有划分的详细结果
    detailed_results = []
    
    for i, (train_years, test_year) in enumerate(train_test_splits, 1):
        
        # 准备数据
        X_train, y_train, X_test, y_test, X_test_orig, y_test_orig = prepare_year_data(
            data, train_years, test_year
        )
        
        # 评估模型
        results = time_series_evaluation(X_train, y_train, X_test, y_test, model_name)
        
        if results is not None:
            yearly_results[f"{train_years[0]}-{train_years[-1]}_to_{test_year}"] = results
            
            # 存储详细结果用于后续汇总
            detailed_results.append({
                'split': f"{train_years[0]}-{train_years[-1]}_to_{test_year}",
                'r2': results['r2'],
                'mse': results['mse'],
                'rmse': results['rmse'],
                'mae': results['mae'],
                'n_train': results['n_train'],
                'n_test': results['n_test'],
            })
            
            print(f"R²: {results['r2']:.4f}, MSE: {results['mse']:.4f}, RMSE: {results['rmse']:.4f}, MAE: {results['mae']:.4f}")
        else:
            print("该划分无法评估")
    
    # 输出每种划分的详细结果
    print(f"\n{'='*80}")
    print("每种训练-测试划分的详细结果:")
    print(f"{'划分':<25} {'R²':<8} {'MSE':<10} {'RMSE':<8} {'MAE':<8} {'训练样本':<10} {'测试样本':<10}")
    print("-" * 80)
    
    for result in detailed_results:
        print(f"{result['split']:<25} {result['r2']:.4f}  {result['mse']:.4f}    {result['rmse']:.4f}   {result['mae']:.4f}   {result['n_train']:<10} {result['n_test']:<10}")
    
    # 计算整体平均性能
    if detailed_results:
        mean_r2 = np.mean([result['r2'] for result in detailed_results])
        mean_mse = np.mean([result['mse'] for result in detailed_results])
        mean_rmse = np.mean([result['rmse'] for result in detailed_results])
        mean_mae = np.mean([result['mae'] for result in detailed_results])
        
        print(f"\n{'='*80}")
        print("整体平均性能汇总:")
        print(f"{'R²':<8} {mean_r2:.4f}")
        print(f"{'MSE':<8} {mean_mse:.4f}")
        print(f"{'RMSE':<8} {mean_rmse:.4f}")
        print(f"{'MAE':<8} {mean_mae:.4f}")
        
        print(f"\n有效划分数量: {len(detailed_results)}")
    
    return yearly_results, detailed_results

def main():
    """
    主函数
    """
    # 加载数据
    data = load_and_prepare_data()
    
    # 进行时间序列分析
    print("\n开始时间序列分析...")
    
    # 使用OLS模型
    ols_results, detailed_results = yearly_time_series_analysis(data, model_name='OLS')
    
    return ols_results, detailed_results

if __name__ == "__main__":
    results, detailed_results = main()


In [ ]:
# 整体预测性能评估方案二：分模型评估
# 由上面模型的数据呈现结果
# 在“使用”部分更改数据

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker  # 引入刻度控制库

# ==========================================
# 全局字体设置：将所有图表字体改为 Times New Roman
# ==========================================
plt.rcParams['font.family'] = 'Times New Roman'
# plt.rcParams['axes.unicode_minus'] = False # 如果遇到负号显示为方块，可取消注释此行

def plot_r2_boxplot_from_data(gbdt_r2_values, rf_r2_values, ols_r2_values):
    """
    从数据直接绘制三种模型R²的箱线图
    改进点：Y轴自适应可见数据范围（排除被隐藏的异常值），并强制0.1刻度间隔
    """
    # 准备数据
    r2_data = []
    
    for r2 in gbdt_r2_values:
        r2_data.append({'Model': 'GBDT', 'R²': r2})
    
    for r2 in rf_r2_values:
        r2_data.append({'Model': 'RF', 'R²': r2})
    
    for r2 in ols_r2_values:
        r2_data.append({'Model': 'OLS', 'R²': r2})
    
    r2_df = pd.DataFrame(r2_data)
    
    # 绘制箱线图
    plt.figure(figsize=(10, 6))
    colors = ['red', 'orange', 'yellow']
    
    # showfliers=False 隐藏异常值
    box_plot = sns.boxplot(data=r2_df, x='Model', y='R²', palette=colors, showfliers=False)
    
    # 计算每个模型的平均值并绘制
    means = []
    for model in ['GBDT', 'RF', 'OLS']:
        model_data = r2_df[r2_df['Model'] == model]['R²']
        if len(model_data) > 0:
            mean_val = model_data.mean()
            means.append(mean_val)
        else:
            means.append(None)
    
    for i, mean_val in enumerate(means):
        if mean_val is not None:
            plt.plot(i, mean_val, 'o', 
                     markersize=6, 
                     markeredgewidth=1, 
                     markeredgecolor='black',
                     markerfacecolor='white')
    
    # ---------------------------------------------------------
    # Y轴自适应设置
    # ---------------------------------------------------------
    
    Q1 = r2_df['R²'].quantile(0.25)
    Q3 = r2_df['R²'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # 筛选出位于须内的可见数据
    visible_data = r2_df[(r2_df['R²'] >= lower_bound) & (r2_df['R²'] <= upper_bound)]
    
    if not visible_data.empty:
        y_min_visible = visible_data['R²'].min()
        y_max_visible = visible_data['R²'].max()

        padding = 0.05
        plt.ylim(y_min_visible - padding, y_max_visible + padding)
    
    # 强制刻度间隔为 0.1
    ax = plt.gca()
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
    
    # ---------------------------------------------------------
    
    # 设置标题和标签
    plt.xlabel('Model', fontsize=12)
    plt.ylabel('R²', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.title('Time-Series Dispersion of R² - R&D Cost', fontsize=14, pad=15)
    
    plt.tight_layout()
    plt.show()
    
    # 返回统计信息
    stats = {}
    for model in ['GBDT', 'RF', 'OLS']:
        model_data = r2_df[r2_df['Model'] == model]['R²']
        if len(model_data) > 0:
            stats[model] = {
                'mean': np.mean(model_data),
                'median': np.median(model_data),
                'std': np.std(model_data),
                'min': np.min(model_data),
                'max': np.max(model_data),
                'count': len(model_data)
            }
    
    return stats

def print_performance_table_from_data(performance_dict):
    """
    从数据直接输出三种模型性能对比表
    """
    print("\n" + "="*80)
    print("三种模型性能对比表")
    print("="*80)
    print(f"{'指标':<10} {'GBDT':<12} {'RF':<12} {'OLS':<12}")
    print("-" * 50)
    
    metrics = ['MSE', 'RMSE', 'MAE', 'R²']
    
    for metric in metrics:
        gbdt_val = performance_dict.get('GBDT', {}).get(metric, 'N/A')
        rf_val = performance_dict.get('RF', {}).get(metric, 'N/A')
        ols_val = performance_dict.get('OLS', {}).get(metric, 'N/A')
        
        # 格式化输出，R²保留4位小数
        if metric == 'MAE':
             print(f"{metric:<10} {gbdt_val:<12.4f} {rf_val:<12.4f} {ols_val:<12.4f}")
        else:
             print(f"{metric:<10} {gbdt_val:<12.4f} {rf_val:<12.4f} {ols_val:<12.4f}")

# 使用
if __name__ == "__main__":
    # 替换为由上面三个模型运行出的实际数据
    # R²值（所有时间划分的结果)
    gbdt_r2 = [0.4572, 0.5275, 0.5068, 0.4925, 0.5707, 0.5896, 0.5791, 0.6192]
    rf_r2 = [0.4965, 0.5126, 0.5059, 0.4766, 0.5524, 0.5407, 0.5214, 0.5753]
    ols_r2 = [0.2643, 0.4930, 0.4898, 0.4458, 0.5266, 0.5362, 0.5069, 0.5465]

    # 性能指标（平均值）
    performance_data = {
        'GBDT': {
            'MSE': np.mean([0.0011, 0.0012, 0.0014, 0.0015, 0.0013, 0.0017, 0.0023, 0.0022]),
            'RMSE': np.mean([0.0330, 0.0348, 0.0372, 0.0391, 0.0362, 0.0411, 0.0481, 0.0467]),
            'MAE': np.mean([0.0213, 0.0218, 0.0230, 0.0244, 0.0228, 0.0252, 0.0291, 0.0288]),
            'R²': np.mean([0.4572, 0.5275, 0.5068, 0.4925, 0.5707, 0.5896, 0.5791, 0.6192])
        },
        'RF': {
            'MSE': np.mean([0.0010, 0.0012, 0.0014, 0.0016, 0.0014, 0.0019, 0.0026, 0.0024]),
            'RMSE': np.mean([0.0318, 0.0354, 0.0372, 0.0397, 0.0369, 0.0435, 0.0513, 0.0494]),
            'MAE': np.mean([0.0203, 0.0219, 0.0235, 0.0247, 0.0235, 0.0260, 0.0306, 0.0305]),
            'R²': np.mean([0.4965, 0.5126, 0.5059, 0.4766, 0.5524, 0.5407, 0.5214, 0.5753])
        },
        'OLS': {
            'MSE': np.mean([0.0015, 0.0013, 0.0014, 0.0017, 0.0014, 0.0019, 0.0027, 0.0026]),
            'RMSE': np.mean([0.0384, 0.0361, 0.0378, 0.0408, 0.0380, 0.0437, 0.0521, 0.0510]),
            'MAE': np.mean([0.0268, 0.0240, 0.0247, 0.0258, 0.0247, 0.0275, 0.0322, 0.0329]),
            'R²': np.mean([0.2643, 0.4930, 0.4898, 0.4458, 0.5266, 0.5362, 0.5069, 0.5465])
        }
    }
    
    # 1. 绘制箱线图
    print("1. 三种模型样本外R²箱线图")
    stats = plot_r2_boxplot_from_data(gbdt_r2, rf_r2, ols_r2)
    
    # 输出统计信息
    print("\nR²统计信息:")
    for model, stat in stats.items():
        print(f"{model}: 均值={stat['mean']:.4f}, 中位数={stat['median']:.4f}, 标准差={stat['std']:.4f}, 范围=[{stat['min']:.4f}, {stat['max']:.4f}]")
    
    # 2. 输出性能对比表
    print("\n2. 三种模型性能对比表")
    print_performance_table_from_data(performance_data)

In [ ]:
#Shap方法特征重要性排名、单特征作用机制分析、特征组相对重要性分析及特征随时间重要性变化分析

import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score
import math

# ==========================================
# 全局字体设置：将所有图表字体改为 Times New Roman
# ==========================================
plt.rcParams['font.family'] = 'Times New Roman'
# plt.rcParams['axes.unicode_minus'] = False # 如果遇到负号显示为方块，可取消注释此行

# ==========================================
# 1. 数据配置与加载
# ==========================================
file_path = '../data/MERGED_DATA_Winsor.dta'

# 特征列表 (X)
feature_cols = [
    'TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG',
    'CEORisk', 'CEODegree', 'IsMBA', 'FinBack', 'CEOAcademicBack', 'Versatility', 'Top3ManageSumSalary', 'CEOIncentive', 'REM', 'IntControl', 'KV', 'Deficiency',
    'EmployeeTurnover', 'ESOP', 'Paygap', 'SocInsurance', 'EmployeeSalary',
    'InvestorShare', 'Ownstructure', 'MajorShareholders', 'Ownership', 'Litigation', 'VAM',
    'Boardsize', 'IndDirectorRatio', 'FemaleDirector', 'DO', 'DCG', 'AISkill', 'DataSkill', 'ChairAcademicBack',
    'FC', 'BOND_D', 'BOND_P', 'BankHHI', 'Top3Bank', 'BankCR',
    'AnaAttention', 'FinNews', 'NegFinNews', 'FERR', 'FOPT', 
    'Financeavailable', 'Shortavailable', 'Liquidity', 'NCSKEW', 'DUVOL', 'SC',
    'GovSub', 'Dige', 'Gdatact', 'Ipp', 'HTZone', 'EnvSubsidy', 'Enrelcost', 'TaxRate', 'Talent', 'IP_EncImp',
    'HHI', 'Zombie_Ratio',
    'SupplierHHI', 'CustomerHHI'
]

# 目标变量 (Y)
target_col = 'RDCost_t1'

print(f"正在读取数据: {file_path} ...")
try:
    data = pd.read_stata(file_path)
except FileNotFoundError:
    print("错误：找不到文件。请检查路径。")
    raise

# 样本筛选
data = data[data['complete_obs'] == 0].copy()

# 缺失值处理
data[feature_cols] = data[feature_cols].fillna(0)
data[target_col] = data[target_col].fillna(0)

print(f"数据加载完成，有效样本量: {len(data)}")

# ==========================================
# 2. 定义滚动窗口划分
# ==========================================
train_test_splits = [
    ([2012, 2013, 2014], 2016),
    ([2012, 2013, 2014, 2015], 2017),
    ([2012, 2013, 2014, 2015, 2016], 2018),
    ([2012, 2013, 2014, 2015, 2016, 2017], 2019),
    ([2012, 2013, 2014, 2015, 2016, 2017, 2018], 2020),
    ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], 2021),
    ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020], 2022),
    ([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021], 2023)
]

all_test_X = []
all_test_shap_values = []
r2_scores = []

gbdt_param_grid = {
    'learning_rate': [0.1, 0.2],
    'n_estimators': [100, 150],
    'max_depth': [3, 5]
}

# ==========================================
# 3. 滚动训练循环
# ==========================================
print("\n开始滚动窗口训练与 SHAP 计算...")

if 'year' not in data.columns:
    raise ValueError("数据集中缺少 'year' 列，无法进行时间窗口划分！")

for train_years, test_year in train_test_splits:
    print(f"\n>>> 处理窗口: 训练集 {train_years} -> 测试集 {test_year}")
    
    X_train = data.loc[data['year'].isin(train_years), feature_cols]
    y_train = data.loc[data['year'].isin(train_years), target_col]
    X_test = data.loc[data['year'] == test_year, feature_cols]
    y_test = data.loc[data['year'] == test_year, target_col]
    
    if len(X_test) == 0:
        print(f"警报：{test_year} 年没有测试数据，跳过。")
        continue

    gbdt = GradientBoostingRegressor(random_state=42)
    grid_search = GridSearchCV(gbdt, gbdt_param_grid, cv=3, scoring='r2', n_jobs=-1)
    grid_search.fit(X_train, y_train)
    
    best_model = grid_search.best_estimator_
    
    y_pred = best_model.predict(X_test)
    score = r2_score(y_test, y_pred)
    r2_scores.append(score)
    print(f"   最佳参数: {grid_search.best_params_}")
    print(f"   测试集 R2: {score:.4f}")

    explainer = shap.TreeExplainer(best_model)
    shap_values_test = explainer.shap_values(X_test)
    
    all_test_X.append(X_test)
    all_test_shap_values.append(shap_values_test)

print(f"\n滚动训练结束。平均测试集 R2: {np.mean(r2_scores):.4f}")

# ==========================================
# 4. 结果聚合
# ==========================================
if not all_test_X:
    raise ValueError("没有收集到任何测试数据。")

combined_X = pd.concat(all_test_X, axis=0)
combined_shap_values = np.vstack(all_test_shap_values)

print(f"聚合完成。总样本量 (2016-2023): {len(combined_X)}")

# ==========================================
# 5. 可视化与重要性分析
# ==========================================

# -------------------------------------------------------------------------
# 5.1 全局重要性排序 (RVI Scores Table with Grouping)
# -------------------------------------------------------------------------
print("\n正在生成 Relative Variable Importance (RVI) 表格...")

# 1. 计算所有特征的 Mean(|SHAP|)
raw_importance = np.abs(combined_shap_values).mean(axis=0)
feature_names = combined_X.columns.tolist()

# 创建初始重要性字典
import_dict = dict(zip(feature_names, raw_importance))

# 2. 定义需要打包的基本面特征
fundamental_features_list = [
    'TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG'
]

# 3. 执行打包逻辑
fundamental_score = 0
final_features_dict = {}

# 遍历所有特征，区分基本面特征和其他特征
for feat, score in import_dict.items():
    if feat in fundamental_features_list:
        fundamental_score += score  # 累加基本面特征的重要性
    else:
        final_features_dict[feat] = score # 其他特征保持不变

# 将打包后的基本面特征加入字典
final_features_dict['Fundamental_Features'] = fundamental_score

# 4. 计算 RVI Scores (归一化到 100%)
total_score = sum(final_features_dict.values())
rvi_dict = {k: (v / total_score) * 100 for k, v in final_features_dict.items()}

# 5. 排序并取前10
sorted_rvi = sorted(rvi_dict.items(), key=lambda x: x[1], reverse=True)
top_10_rvi = sorted_rvi[:10]

# 6. 输出表格
print("-" * 50)
print(f"{'Rank':<5} | {'Variable':<25} | {'RVI Scores(%)':<15}")
print("-" * 50)
for rank, (var, score) in enumerate(top_10_rvi, 1):
    print(f"{rank:<5} | {var:<25} | {score:.2f}")
print("-" * 50)


# -------------------------------------------------------------------------
# 5.2 作用机制图 (Dependence Plot with Mean Effect Line)
# -------------------------------------------------------------------------
print("\n正在绘制单个重要特征的作用机制图 (Top 20 Individual Features)...")

# 重新计算原始特征的重要性排序（不打包）
mean_abs_shap_raw = np.abs(combined_shap_values).mean(axis=0)
top_indices_raw = np.argsort(mean_abs_shap_raw)[::-1][:20] 
top_features_raw = [combined_X.columns[i] for i in top_indices_raw]

for feature_name in top_features_raw:
    fig, ax = plt.subplots(figsize=(8, 6))
    
    # A. 绘制散点
    shap.dependence_plot(
        feature_name, 
        combined_shap_values, 
        combined_X, 
        interaction_index=None, 
        ax=ax,
        show=False,
        alpha=0.2,       
        dot_size=15      
    )
    
    # B. 添加平滑趋势线 (Rolling Mean)
    x_data = combined_X[feature_name].values
    y_data = combined_shap_values[:, combined_X.columns.get_loc(feature_name)]
    
    # 1. 排序
    sort_idx = np.argsort(x_data)
    x_sorted = x_data[sort_idx]
    y_sorted = y_data[sort_idx]

    # 2. 计算移动平均
    window_size = int(len(x_sorted) * 0.05) 
    if window_size < 1: window_size = 1 

    y_rolling = pd.Series(y_sorted).rolling(window=window_size, center=True).mean()

    # 3. 绘制折线
    ax.plot(x_sorted, y_rolling, color='black', linewidth=2.5, label='Mean Effect')
    
    # 参考线
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

    #设置图标题
    ax.set_title(f"{feature_name} -> Invention Patent", fontsize=12)
    ax.set_xlabel(f"Feature Value: {feature_name}", fontsize=10)
    ax.set_ylabel("SHAP Value", fontsize=10)
    ax.legend()
    
    plt.tight_layout()
    plt.show()

print("所有分析完成。")

# =========================================================================
# 5.2.1 综合子图版：指定核心特征作用机制图 (Multi-subplot Dependence Plot)
# =========================================================================
print("\n正在绘制指定特征的综合作用机制大图...")

# 1. 在这里填入想要放在一张图里展示的所有特定特征
target_features = [
    'EmployeeSalary', 'GovSub', 'HHI', 'DCG', 'TaxRate', 
    'AISkill', 'AnaAttention', 'REM', 'EmployeeTurnover'
]

# 过滤掉可能拼错或不在数据集中的特征，防止报错
valid_features = [f for f in target_features if f in combined_X.columns]
n_features = len(valid_features)

if n_features == 0:
    print("警告：指定的特征均不在数据集中，请检查特征名称。")
else:
    # 2. 动态设置网格布局 (默认每行画 3 个图，您可以改成 4)
    cols = 3
    rows = math.ceil(n_features / cols)
    
    # 根据行列数动态分配整张大画布的尺寸 (每个子图宽 6，高 5)
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    
    # 如果只有一个子图或单行单列，axes 可能不是一维数组，需要展平处理
    if n_features > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    # 3. 循环遍历指定的特征并绘制在对应的子图(ax)上
    for i, feature_name in enumerate(valid_features):
        ax = axes[i]
        
        # A. 绘制 SHAP 散点图
        shap.dependence_plot(
            feature_name, 
            combined_shap_values, 
            combined_X, 
            interaction_index=None, 
            ax=ax,
            show=False,       # 必须为 False，否则会中途跳出弹窗
            alpha=0.2,        
            dot_size=15      
        )
        
        # B. 添加平滑趋势线 (Rolling Mean)
        x_data = combined_X[feature_name].values
        # 获取特征在矩阵中的列索引
        feature_idx = combined_X.columns.get_loc(feature_name)
        y_data = combined_shap_values[:, feature_idx]
        
        # 排序
        sort_idx = np.argsort(x_data)
        x_sorted = x_data[sort_idx]
        y_sorted = y_data[sort_idx]

        # 计算移动平均 (滑动窗口为 5%)
        window_size = int(len(x_sorted) * 0.05) 
        if window_size < 1: window_size = 1 
        y_rolling = pd.Series(y_sorted).rolling(window=window_size, center=True).mean()

        # 绘制黑色的趋势折线
        ax.plot(x_sorted, y_rolling, color='black', linewidth=2.5, label='Mean Effect')
        
        # 添加 0 轴基准线
        ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

        # C. 设置单个子图的标题和坐标标签
        ax.set_title(f"{feature_name}", fontsize=13, fontweight='bold')
        ax.legend(loc='best')
        
    # 4. 删除因为行列除不尽而多出来的空白子图
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # 5. 设置整张大图的总标题和自动排版
    plt.suptitle("SHAP Dependence Plots - R&D Cost", fontsize=20, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

print("多子图合并绘制完成。")

   

# ==========================================
# 5.3 特征组相对重要性分析 (Feature Group RVI with Top Feature)
# ==========================================
print("\n正在进行特征组 RVI 分析与绘图...")

# 1. 定义特征组
raw_groups = {
    'Fundamental': [ 
        'TotalAssets', 'Age', 'DebtRatio', 'Cashflow', 'Growth', 'ProfitMargin', 'ROA', 'FixedAssetRatio', 'EV', 'OverseasInv', 'OverseasRev', 'MA', 'ESG'
    ],
    'Industry': [
        'HHI', 'Zombie_Ratio'
    ],
    'Manager': [
        'CEORisk', 'CEODegree', 'IsMBA', 'FinBack', 'CEOAcademicBack', 'Versatility', 'Top3ManageSumSalary', 'CEOIncentive', 'REM', 'IntControl', 'KV', 'Deficiency'
    ],
    'Employee': [
        'EmployeeTurnover', 'ESOP', 'Paygap', 'SocInsurance', 'EmployeeSalary'
    ],
    'Shareholder': [
        'InvestorShare', 'Ownstructure', 'MajorShareholders', 'Ownership', 'Litigation', 'VAM'
    ],
    'Board': [
        'Boardsize', 'IndDirectorRatio', 'FemaleDirector', 'DO', 'DCG', 'AISkill', 'DataSkill', 'ChairAcademicBack'
    ],
    'Creditor': [
        'FC', 'BOND_D', 'BOND_P', 'BankHHI', 'Top3Bank', 'BankCR'
    ],    
    'Government': [
        'GovSub', 'Dige', 'Gdatact', 'Ipp', 'HTZone', 'EnvSubsidy', 'Enrelcost', 'TaxRate', 'Talent', 'IP_EncImp'
    ],
    'Infomediary': [
        'AnaAttention', 'FinNews', 'NegFinNews', 'FERR', 'FOPT'
    ],
    'Stock Market': [
        'Financeavailable', 'Shortavailable', 'Liquidity', 'NCSKEW', 'DUVOL', 'SC'
    ],
    'Supply Chain': [
        'SupplierHHI', 'CustomerHHI'
    ]
}

feature_groups = {}
valid_features = set(feature_cols)

for grp_name, feat_list in raw_groups.items():
    valid_list = [f for f in feat_list if f in valid_features]
    if valid_list:
        feature_groups[grp_name] = valid_list

# 2. 计算全局 RVI (Global Importance)
global_mean_abs_shap = np.abs(combined_shap_values).mean(axis=0)
global_import_dict = dict(zip(feature_cols, global_mean_abs_shap))
total_global_importance = sum(global_import_dict.values())

group_data = []

for grp, feats in feature_groups.items():
    # A. 组总分
    grp_score_raw = sum(global_import_dict[f] for f in feats)
    grp_rvi = (grp_score_raw / total_global_importance) 
    
    # B. 组内最强特征
    max_feat = max(feats, key=lambda f: global_import_dict[f])
    max_feat_score_raw = global_import_dict[max_feat]
    max_feat_rvi = (max_feat_score_raw / total_global_importance)
    
    group_data.append({
        'Group': grp,
        'Group_RVI': grp_rvi,
        'Top_Feature': max_feat,
        'Top_Feature_RVI': max_feat_rvi
    })

# 转为 DataFrame 并排序
df_group = pd.DataFrame(group_data).sort_values('Group_RVI', ascending=False)

# 4. 计算年度 RVI 散点 (Yearly Scatter Points)
# 利用 all_test_shap_values 和 all_test_X 中的年份信息（如果有）
# 或者利用 train_test_splits 的结构 (test_year 对应 list index)

yearly_points = {grp: [] for grp in df_group['Group']}

# 遍历每一个测试年份
for i, shap_matrix in enumerate(all_test_shap_values):
    # 当前年份的总重要性
    year_mean_abs = np.abs(shap_matrix).mean(axis=0)
    year_dict = dict(zip(feature_cols, year_mean_abs))
    year_total = sum(year_dict.values())
    
    # 计算该年份各组的 RVI
    for grp in df_group['Group']:
        feats = feature_groups[grp]
        grp_score = sum(year_dict[f] for f in feats)
        # 归一化到该年的 0-1 之间
        yearly_points[grp].append(grp_score / year_total)

# ==========================================
# 绘图
# ==========================================
import matplotlib.cm as cm
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

# 调整画布大小
fig, ax = plt.subplots(figsize=(20, 8))

# 1. 设置颜色映射
colors = cm.rainbow(np.linspace(1, 0, len(df_group)))

# 2. 定义柱子位置与宽度
x_pos = np.arange(len(df_group))
bar_width = 0.35

# 3. 绘制柱状图
# (1) 特征组重要性
bars_group = ax.bar(x_pos - bar_width/2, df_group['Group_RVI'], 
                    width=bar_width, color=colors, alpha=0.85, label='Group Importance (Total)', zorder=1)

# (2) 组内最强特征
bars_top = ax.bar(x_pos + bar_width/2, df_group['Top_Feature_RVI'], 
                  width=bar_width, color=colors, alpha=0.4, label='Top Feature (Individual)', zorder=1)

# 4. 绘制年度散点 & 添加标签
for idx, row in enumerate(df_group.itertuples()):
    grp_name = row.Group
    
    # --- A. 绘制年度散点 (Yearly Scatters) ---
    # 保持在左侧柱子中心
    points = yearly_points[grp_name]
    x_scatter = [x_pos[idx] - bar_width/2] * len(points)
    ax.scatter(x_scatter, points, color=colors[idx], s=35, alpha=0.6, zorder=2, edgecolor='none')
    
    # --- B. 添加最强特征名字标签 ---
    ax.text(x_pos[idx], row.Top_Feature_RVI + 0.002, row.Top_Feature, 
            ha='left', va='bottom', fontsize=8, rotation=0, color='black')

# 5. 坐标轴与图例设置
ax.set_xticks(x_pos)
ax.set_xticklabels(df_group['Group'], rotation=45, ha='right', fontsize=12)
ax.set_ylabel('Normalized SHAP Importance (RVI)', fontsize=12)
ax.set_title('Feature Group Importance with Top Feature - R&D Cost', fontsize=16, fontweight='bold')

# 添加虚线网格
ax.grid(axis='y', linestyle='--', alpha=0.3)

# 自定义图例
legend_elements = [
    Patch(facecolor='red', alpha=0.85, label='Group Total Importance'),
    Patch(facecolor='red', alpha=0.4, label='Top Feature in Group'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=8, label='Yearly RVI (Scatter)')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()


# ==========================================
# 5.4 特征随时间重要性变化分析 (Temporal Trend Analysis) - Top 5 绘图版
# ==========================================
print("\n" + "="*50)
print("正在进行特征随时间重要性变化趋势分析...")
print("="*50)

# 1. 构建年度 RVI DataFrame
years = [split[1] for split in train_test_splits]

if len(years) != len(all_test_shap_values):
    print("警告：年份数量与 SHAP 值列表长度不一致，趋势分析可能不准确。")

rvi_data = {}
for i, year in enumerate(years):
    if i < len(all_test_shap_values):
        shap_vals = all_test_shap_values[i]
        mean_abs_shap = np.abs(shap_vals).mean(axis=0)
        
        # 归一化为 RVI (占比)
        total = mean_abs_shap.sum()
        if total == 0: total = 1 
        rvi = mean_abs_shap / total 
        
        rvi_data[year] = rvi

# 行：特征，列：年份
rvi_df = pd.DataFrame(rvi_data, index=feature_cols)

# ==========================================
# A. 绘制年度 RVI 矩阵热力图
# ==========================================
print("正在绘制全变量年度 RVI 矩阵热力图 (增强对比度版)...")
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
# 1. 数据准备：按平均 RVI 降序排列
sorted_idx = rvi_df.mean(axis=1).sort_values(ascending=False).index
rvi_df_sorted = rvi_df.loc[sorted_idx]

sorted_rvi_matrix = rvi_df_sorted.values
sorted_features = rvi_df_sorted.index.tolist()
years_str = [str(y) for y in years]

# 2. 动态调整画布高度
dynamic_height = max(10, len(sorted_features) * 0.25)
fig, ax1 = plt.subplots(figsize=(12, dynamic_height))

# 3. 配色方案增强
try:
    blues_enhanced = matplotlib.colormaps['Blues'].resampled(256).copy()
except:
    blues_enhanced = matplotlib.cm.get_cmap('Blues').copy()

blues_enhanced.set_gamma(0.5) 

# 4. 计算颜色范围
vmin_percentile = np.percentile(sorted_rvi_matrix, 5)
vmax_percentile = np.percentile(sorted_rvi_matrix, 95)

# 5. 绘制热力图
im1 = ax1.imshow(sorted_rvi_matrix, aspect='auto', cmap=blues_enhanced, 
                 vmin=vmin_percentile, vmax=vmax_percentile)

# 6. 设置标签
ax1.set_yticks(range(len(sorted_features)))
ax1.set_yticklabels(sorted_features, fontsize=9)
ax1.set_xticks(range(len(years)))
ax1.set_xticklabels(years_str, fontsize=11, rotation=45)
ax1.set_title('Time Variability of the Variable Importance - R&D Cost', fontsize=15, fontweight='bold', pad=20)

# 7. 添加精细网格线
ax1.set_xticks(np.arange(-0.5, len(years), 1), minor=True)
ax1.set_yticks(np.arange(-0.5, len(sorted_features), 1), minor=True)
ax1.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.3)
ax1.tick_params(which='minor', size=0)

# 8. 添加颜色条
cbar1 = plt.colorbar(im1, ax=ax1, pad=0.02)
cbar1.set_label('Normalized RVI Score', fontsize=12)

plt.tight_layout()
plt.show()

# ==========================================
# B. 执行趋势回归 (OLS) & 计算排名与稳定性指标
# ==========================================
trend_results = []
X_time = np.arange(len(years)) 
X_time_const = sm.add_constant(X_time)

# 计算特征在每年的排名 (axis=0, 1为最高)
rank_df = rvi_df.rank(axis=0, ascending=False)

for feature in rvi_df.index:
    y_rvi = rvi_df.loc[feature].values
    
    if np.var(y_rvi) < 1e-10:
        continue

    try:
        model = sm.OLS(y_rvi, X_time_const).fit()
        
        slope = model.params[1]
        p_value = model.pvalues[1]
        r_squared = model.rsquared
        
        start_rvi_fitted = model.params[0] + model.params[1] * X_time[0]
        end_rvi_fitted = model.params[0] + model.params[1] * X_time[-1]
        
        if start_rvi_fitted <= 1e-6: start_rvi_fitted = 1e-6
        change_pct = (end_rvi_fitted - start_rvi_fitted) / start_rvi_fitted
        
        avg_position = rank_df.loc[feature].mean()

        # 【核心新增】：计算变异系数 CV (标准差 / 均值) 来刻画跨期稳定性
        mean_rvi = y_rvi.mean()
        std_rvi = np.std(y_rvi)
        cv = std_rvi / mean_rvi if mean_rvi > 0 else 0

        trend_results.append({
            'Feature': feature,
            'Slope': slope,
            'P_Value': p_value,
            'R2': r_squared,
            'Change_Pct': change_pct,
            'Start_RVI': start_rvi_fitted,
            'End_RVI': end_rvi_fitted,
            'Mean_RVI': mean_rvi,
            'Std_RVI': std_rvi,
            'CV': cv,            # 变异系数（越小越稳定）
            'Avg_Position': avg_position 
        })
    except Exception as e:
        continue

trend_df = pd.DataFrame(trend_results)

# ==========================================
# C. 输出文本报告
# ==========================================
# 1. 筛选核心稳定特征 (逻辑升级：在最重要的前10名中，挑选波动系数 CV 最小的 5 个)
top_10_important = trend_df.sort_values('Mean_RVI', ascending=False).head(10)
core_stable_df = top_10_important.sort_values('CV', ascending=True).head(5)

# 2. 筛选显著变化特征 (P < 0.1)
sig_change_df = trend_df[trend_df['P_Value'] < 0.1].copy()

# 【新兴特征过滤】：起点 < 0.01，按绝对增长斜率(Slope)排序
rising_df = sig_change_df[
    (sig_change_df['Slope'] > 0) & 
    (sig_change_df['Start_RVI'] < 0.01)
].sort_values('Slope', ascending=False)

falling_df = sig_change_df[sig_change_df['Slope'] < 0].sort_values('Slope', ascending=True)

print("\n" + "="*90)
print("RVI分数变化模式分析报告 (引入 CV 稳定性验证版)")
print("="*90)

# (1) 核心稳定特征输出
print("\n1. 核心稳定特征 (Top 10 重要特征中，跨期变异系数 CV 最小的 Top 5):")
print(f"{'排名':<4} {'特征':<25} {'平均RVI':<10} {'变异系数(CV)':<12} {'(参考: 平均排名)'}")
print("-" * 75)
for i, (idx, row) in enumerate(core_stable_df.iterrows(), 1):
    print(f"{i:<4} {row['Feature']:<25} {row['Mean_RVI']:<10.4f} {row['CV']:<12.4f} ({row['Avg_Position']:.1f})")

# (2) 显著变化特征输出
print(f"\n2. RVI分数显著变化的特征 (p-value < 0.1):")
print(f"  总共发现 {len(sig_change_df)} 个显著变化的特征")
print(f"  其中: {len(rising_df)} 个上升特征, {len(falling_df)} 个下降特征")

def print_trend_table(title, df):
    if df.empty:
        print(f"\n  {title}: (无)")
        return
    print(f"\n  {title}:")
    header = f"{'序号':<4} {'特征':<25} {'增长斜率':<12} {'p值':<8} {'R²':<8} {'起始RVI':<10} {'终止RVI':<10}"
    print(header)
    print("-" * 90)
    for i, (idx, row) in enumerate(df.iterrows(), 1):
        print(f"{i:<4} {row['Feature']:<25} {row['Slope']:>+10.6f}    {row['P_Value']:.3f}    {row['R2']:.3f}     {row['Start_RVI']:.4f}     {row['End_RVI']:.4f}")

print_trend_table("显著上升的新兴特征 (起点<1%，按增长斜率排序)", rising_df)
print_trend_table("显著下降的特征 (按绝对下降斜率排序)", falling_df)
print("-" * 90)

# ==========================================
# D. 绘图: Temporal Evolution (仅呈现显著上升的 Top 5 特征)
# ==========================================
print("正在绘制显著上升特征趋势图 (Top 5)...")

# --- 截取前 5 个特征 ---
top_rising_plot = rising_df.head(5)

if not top_rising_plot.empty:
    plt.figure(figsize=(12, 7))
    
    import matplotlib.cm as cm
    
    # 使用 tab10 色盘，保证颜色区分度高
    cmap = plt.get_cmap('tab10')

    for i, (idx, row) in enumerate(top_rising_plot.iterrows()):
        feat = row['Feature']
        color = cmap(i % cmap.N)
        
        plt.plot(years, rvi_df.loc[feat], marker='o', linestyle='-', linewidth=2.5, 
                 color=color, label=feat)

    plt.title('Temporal Evolution of Top 5 Rising Features - R&D Cost', fontsize=15, fontweight='bold')
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('RVI Score (Relative Importance)', fontsize=12)
    
    # 图例
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., fontsize=10, title="Top 5 Rising Features")
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print("未发现显著上升的特征，跳过绘图。")


# ==========================================
# 5.5 指定多个核心特征的交互作用分析
# ==========================================
print("\n" + "="*80)
print("正在计算指定核心特征的交互作用 (SHAP Interaction Values) 及其解释占比...")
print("="*80)

target_interact_vars = ['EmployeeSalary', 'GovSub', 'HHI', 'DCG', 'TaxRate', 'AISkill', 'AnaAttention', 'REM', 'EmployeeTurnover']  # 指定核心变量

valid_targets = [var for var in target_interact_vars if var in combined_X.columns]
if set(target_interact_vars) - set(valid_targets):
    print(f"警告：跳过不在特征列表中的变量: {set(target_interact_vars) - set(valid_targets)}")

if valid_targets:
    explainer_interact = shap.TreeExplainer(best_model)
    
    print("正在计算底层交互矩阵，请耐心等待...")
    shap_interactions = explainer_interact.shap_interaction_values(combined_X)

    # 1. 计算交互特征的绝对平均重要性矩阵 (n_features, n_features)
    mean_abs_interaction = np.abs(shap_interactions).mean(axis=0)
    features = combined_X.columns.tolist()

    # 2. 提前计算各个特征的“总 SHAP 重要性”和“全局模型总重要性”
    # np.abs(combined_shap_values).mean(axis=0) 等价于每个特征的总解释力
    feature_total_importance = np.abs(combined_shap_values).mean(axis=0)
    global_total_importance = feature_total_importance.sum()

    for target_var in valid_targets:
        print(f"\n" + "-"*80)
        print(f"正在深入分析核心变量: 【{target_var}】")
        print("-" * 80)
        
        target_idx = features.index(target_var)
        target_total_shap = feature_total_importance[target_idx] # 获取变量1的总解释力

        interaction_records = []
        for j, feature_j in enumerate(features):
            if j == target_idx:
                continue 
            
            # 交互总分 (双向之和)
            interaction_score = mean_abs_interaction[target_idx, j] * 2
            
            # 占比计算 1：归属于变量1的交互份额 / 变量1总重要性
            pct_of_var1 = (mean_abs_interaction[target_idx, j] / target_total_shap) * 100
            
            # 占比计算 2：交互总分 / 模型全局所有变量总重要性
            pct_of_global = (interaction_score / global_total_importance) * 100
            
            interaction_records.append({
                'Variable1': target_var,
                'Variable2': feature_j,
                'Score': interaction_score,
                'Pct_of_Var1': pct_of_var1,
                'Pct_of_Global': pct_of_global
            })

        interaction_df = pd.DataFrame(interaction_records)
        interaction_df = interaction_df.sort_values(by='Score', ascending=False).reset_index(drop=True)

        top_10 = interaction_df.head(10)
        top_10.index = np.arange(1, len(top_10) + 1)

        print(f"\n[{target_var}] 的 Top 10 最强交互对:")
        print(f"{'Rank':<4} | {'Variable1':<18} | {'Variable2':<18} | {'Score':<8} | {'% of Var1':<10} | {'% of Global':<10}")
        print("-" * 82)
        for rank, row in top_10.iterrows():
            print(f"{rank:<4} | {row['Variable1']:<18} | {row['Variable2']:<18} | {row['Score']:<8.4f} | {row['Pct_of_Var1']:>7.2f}% | {row['Pct_of_Global']:>8.2f}%")
        
        top_v2 = top_10.iloc[0]['Variable2']
        pct_v1_display = top_10.iloc[0]['Pct_of_Var1']
        
        print(f"\n>> 发现最强交互项：{top_v2} (此交互效应占 {target_var} 自身总解释力的 {pct_v1_display:.2f}%)")
        print(f">> 正在绘制 [{target_var}] & [{top_v2}] 交互依赖图...")

        fig, ax = plt.subplots(figsize=(8, 6))
        shap.dependence_plot(
            target_var, 
            combined_shap_values, 
            combined_X, 
            interaction_index=top_v2, 
            ax=ax,
            show=False,
            alpha=0.6,
            dot_size=15,
            cmap=plt.get_cmap("coolwarm")
        )
        ax.set_title(f"Interaction: {target_var} & {top_v2}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

print("\n所有指定特征的交互分析全部完成！")